# Map ESCO to ONET for Job Zones (Education)

Merge O*NET crosswalk data with job zones to enable mapping ESCO occupations to O*NET education levels. The job zones in O*NET indicate education and experience requirements.

**Note:** This notebook explores the O*NET mapping process. Future implementation will move reusable logic to `src/`.

This notebook contains the exploration and can be used for testing individual projects.

## 0. Setup

### 0.01 Import Required Libraries

In [52]:
import pandas as pd
from pathlib import Path

# Import our config
import sys
sys.path.append(str(Path.cwd().parent))
from src.config import load_config

### 0.02 Load Configuration

In [53]:
# Get project root
project_root = Path.cwd().parent

# Load project config
config = load_config()

print("✓ Configuration loaded")

✓ Configuration loaded


### 0.03 Set Up Paths

In [54]:
# Project ID to use for this exploration
PROJECT_ID = "P511453"

# Set up paths for O*NET data files
onet_dir = project_root / "data" / "bronze" / "onet"
crosswalk_file = onet_dir / "esco_onet_crosswalk.csv"
job_zones_file = onet_dir / "onet_job_zones.txt"

# Set up paths for unique ESCO NACE file
unique_esco_nace_dir = project_root / "data" / "silver" / "unique_esco_nace_csv"
unique_esco_nace_file = unique_esco_nace_dir / f"{PROJECT_ID}_unique_matched_with_nace.csv"

print(f"Project ID: {PROJECT_ID}")
print(f"✓ Crosswalk file: {crosswalk_file}")
print(f"✓ Job zones file: {job_zones_file}")
print(f"✓ Unique ESCO NACE file: {unique_esco_nace_file}")

Project ID: P511453
✓ Crosswalk file: /Users/lauren/repos/PAD2Skills/data/bronze/onet/esco_onet_crosswalk.csv
✓ Job zones file: /Users/lauren/repos/PAD2Skills/data/bronze/onet/onet_job_zones.txt
✓ Unique ESCO NACE file: /Users/lauren/repos/PAD2Skills/data/silver/unique_esco_nace_csv/P511453_unique_matched_with_nace.csv


## 1. Load and Merge O*NET Data

### 1.01 Load ESCO-ONET Crosswalk

In [55]:
# Load ESCO-ONET crosswalk
crosswalk_df = pd.read_csv(crosswalk_file)

print(f"✓ Loaded {len(crosswalk_df)} crosswalk records")
print(f"  Columns: {', '.join(crosswalk_df.columns)}")
print(f"\nFirst few rows:")
crosswalk_df.head()

✓ Loaded 4253 crosswalk records
  Columns: O*NET Id, O*NET Title, O*NET Description, ESCO or ISCO URI, ESCO or ISCO Title, ESCO or ISCO Description, Type of Match

First few rows:


,O*NET Id,O*NET Title,O*NET Description,ESCO or ISCO URI,ESCO or ISCO Title,ESCO or ISCO Description,Type of Match
0,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,http://data.europa.eu/esco/occupation/5c5b153e...,secretary general,Secretaries general head international governm...,closeMatch
1,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,http://data.europa.eu/esco/occupation/6c3fd65e...,chief executive officer,Chief executive officers hold the highest rank...,exactMatch
2,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,http://data.europa.eu/esco/occupation/c64a6e4e...,chief operating officer,Chief operating officers are the right hand an...,broadMatch
3,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,http://data.europa.eu/esco/occupation/4be4ea31...,airport chief executive,Airport chief executives lead a group of airpo...,broadMatch
4,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,http://data.europa.eu/esco/occupation/73b10c97...,social entrepreneur,Social entrepreneurs create innovative product...,broadMatch


### 1.02 Load O*NET Job Zones

In [56]:
# Load O*NET job zones (tab-separated file)
job_zones_df = pd.read_csv(job_zones_file, sep='\t')

print(f"✓ Loaded {len(job_zones_df)} job zone records")
print(f"  Columns: {', '.join(job_zones_df.columns)}")
print(f"\nFirst few rows:")
job_zones_df.head()

✓ Loaded 923 job zone records
  Columns: O*NET-SOC Code, Job Zone, Date, Domain Source

First few rows:


,O*NET-SOC Code,Job Zone,Date,Domain Source
0,11-1011.00,5,08/2023,Analyst
1,11-1011.03,5,08/2021,Analyst
2,11-1021.00,4,08/2023,Analyst
3,11-1031.00,4,06/2008,Analyst
4,11-2011.00,4,08/2018,Analyst


### 1.03 Merge Crosswalk with Job Zones

In [57]:
# Merge on O*NET ID
# The crosswalk uses "O*NET Id" and job zones uses "O*NET-SOC Code"
onet_merged_df = crosswalk_df.merge(
    job_zones_df,
    left_on='O*NET Id',
    right_on='O*NET-SOC Code',
    how='left'
)

print(f"✓ Merged {len(onet_merged_df)} records")
print(f"  Original crosswalk: {len(crosswalk_df)} records")
print(f"  Job zones: {len(job_zones_df)} records")
print(f"\nMerged columns: {', '.join(onet_merged_df.columns)}")
print(f"\nFirst few rows:")
onet_merged_df.head()

✓ Merged 4253 records
  Original crosswalk: 4253 records
  Job zones: 923 records

Merged columns: O*NET Id, O*NET Title, O*NET Description, ESCO or ISCO URI, ESCO or ISCO Title, ESCO or ISCO Description, Type of Match, O*NET-SOC Code, Job Zone, Date, Domain Source

First few rows:


,O*NET Id,O*NET Title,O*NET Description,ESCO or ISCO URI,ESCO or ISCO Title,ESCO or ISCO Description,Type of Match,O*NET-SOC Code,Job Zone,Date,Domain Source
0,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,http://data.europa.eu/esco/occupation/5c5b153e...,secretary general,Secretaries general head international governm...,closeMatch,11-1011.00,5.0,08/2023,Analyst
1,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,http://data.europa.eu/esco/occupation/6c3fd65e...,chief executive officer,Chief executive officers hold the highest rank...,exactMatch,11-1011.00,5.0,08/2023,Analyst
2,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,http://data.europa.eu/esco/occupation/c64a6e4e...,chief operating officer,Chief operating officers are the right hand an...,broadMatch,11-1011.00,5.0,08/2023,Analyst
3,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,http://data.europa.eu/esco/occupation/4be4ea31...,airport chief executive,Airport chief executives lead a group of airpo...,broadMatch,11-1011.00,5.0,08/2023,Analyst
4,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,http://data.europa.eu/esco/occupation/73b10c97...,social entrepreneur,Social entrepreneurs create innovative product...,broadMatch,11-1011.00,5.0,08/2023,Analyst


### 1.04 Clean and Rename Columns

In [58]:
# Rename columns
onet_merged_df = onet_merged_df.rename(columns={
    'O*NET Id': 'onet_id',
    'O*NET Title': 'onet_title',
    'O*NET Description': 'onet_description',
    'ESCO or ISCO URI': 'uri',
    'Job Zone': 'job_zone',
    'ESCO or ISCO Title': 'esco_title',
    'ESCO or ISCO Description': 'esco_description'
})

# Drop unnecessary columns
columns_to_drop = [
    'Type of Match',
    'O*NET-SOC Code',
    'Date',
    'Domain Source'
]
onet_merged_df = onet_merged_df.drop(columns=columns_to_drop)

print(f"✓ Cleaned dataframe")
print(f"  Remaining columns: {', '.join(onet_merged_df.columns)}")
print(f"\nFirst few rows:")
onet_merged_df.head()

✓ Cleaned dataframe
  Remaining columns: onet_id, onet_title, onet_description, uri, esco_title, esco_description, job_zone

First few rows:


,onet_id,onet_title,onet_description,uri,esco_title,esco_description,job_zone
0,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,http://data.europa.eu/esco/occupation/5c5b153e...,secretary general,Secretaries general head international governm...,5.0
1,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,http://data.europa.eu/esco/occupation/6c3fd65e...,chief executive officer,Chief executive officers hold the highest rank...,5.0
2,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,http://data.europa.eu/esco/occupation/c64a6e4e...,chief operating officer,Chief operating officers are the right hand an...,5.0
3,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,http://data.europa.eu/esco/occupation/4be4ea31...,airport chief executive,Airport chief executives lead a group of airpo...,5.0
4,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,http://data.europa.eu/esco/occupation/73b10c97...,social entrepreneur,Social entrepreneurs create innovative product...,5.0


### 1.05 Extract ESCO ID from URI

In [59]:
# Extract ESCO ID from the URI (last part after the last slash)
onet_merged_df['esco_id'] = onet_merged_df['uri'].str.split('/').str[-1]

# Drop the URI column
onet_merged_df = onet_merged_df.drop(columns=['uri'])

print(f"✓ Extracted ESCO ID")
print(f"  Remaining columns: {', '.join(onet_merged_df.columns)}")
print(f"\nFirst few rows:")
onet_merged_df.head()

✓ Extracted ESCO ID
  Remaining columns: onet_id, onet_title, onet_description, esco_title, esco_description, job_zone, esco_id

First few rows:


,onet_id,onet_title,onet_description,esco_title,esco_description,job_zone,esco_id
0,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,secretary general,Secretaries general head international governm...,5.0,5c5b153e-4bf8-4f3d-973d-12fabf306d12
1,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,chief executive officer,Chief executive officers hold the highest rank...,5.0,6c3fd65e-2d24-47d8-bc22-9e93512bdcc2
2,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,chief operating officer,Chief operating officers are the right hand an...,5.0,c64a6e4e-5b38-4f93-b26d-aded817aeaf3
3,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,airport chief executive,Airport chief executives lead a group of airpo...,5.0,4be4ea31-1211-4f0c-82bb-f6fe10791f4d
4,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,social entrepreneur,Social entrepreneurs create innovative product...,5.0,73b10c97-b003-45dc-a36d-c1d585c04be1


## 2. Merge O*NET onto Unique ESCO Dataset

### 2.01 Load Unique ESCO Occupations with NACE Codes

In [60]:
# Load the unique ESCO occupations with NACE codes
df_occupations = pd.read_csv(unique_esco_nace_file)

print(f"Loaded {len(df_occupations)} unique ESCO occupations")
print(f"Columns: {list(df_occupations.columns)}")
print(f"\nFirst few rows:")
df_occupations.head()

Loaded 44 unique ESCO occupations
Columns: ['esco_id', 'esco_label', 'esco_description', 'group_code', 'group_label_en', 'division_code', 'division_label_en', 'section_code', 'section_label_en', 'pad_occupations', 'pad_activities', 'pad_skills', 'pad_quotes']

First few rows:


,esco_id,esco_label,esco_description,group_code,group_label_en,division_code,division_label_en,section_code,section_label_en,pad_occupations,pad_activities,pad_skills,pad_quotes
0,05ebeb56-6ceb-488d-ba91-ed15088efc8e,bridge construction supervisor,Bridge construction supervisors monitor the co...,421,42.1 Construction of roads and railways,42,42 Civil engineering,F,F CONSTRUCTION,"""construction supervisor""","""Supervise construction and installation of fl...","""construction management"", ""site inspection"", ...","ANNEX 4: CLIMATE RISKS 46: ""To enhance the ..."
1,0752ed49-03e7-4d75-8314-a051b3771a1d,public administration manager,"Public administration managers direct, monitor...",841,84.1 Administration of the State and the econo...,84,84 Public administration and defence; compulso...,P,P PUBLIC ADMINISTRATION AND DEFENCE; COMPULSOR...,"""head of department""","""Be competitively recruited and take part in o...","""performance management"", ""operational leaders...","V. KEY RISKS: ""Operationalization initiated: ..."
2,0ba06640-e0ac-4911-9e43-289a8e41651e,corporate trainer,"Corporate trainers train, coach, and guide emp...",855,85.5 Other education,85,85 Education,Q,Q EDUCATION,"""capacity building specialist""","""Design and implement capacity-building progra...","""training program design"", ""adult learning tec...","V. KEY RISKS: ""Eligible expenditures include: ..."
3,13d1b2b4-99dd-44da-9734-c9f74bae18f7,customer service representative,Customer service representatives handle compla...,822,82.2 Activities of call centres,82,"82 Office administrative, office support and o...",O,O ADMINISTRATIVE AND SUPPORT SERVICE ACTIVITIES,"""grievance officer""","""Operate the National Grievance Redress Mechan...","""grievance intake"", ""case management"", ""record...","IV. PROJECT APPRAISAL SUMMARY: ""The project wi..."
4,16b974f4-cb8c-436c-8c05-a16957c40131,fossil-fuel power plant operator,Fossil-fuel power plant operators operate and ...,351,"35.1 Electric power generation, transmission a...",35,"35 Electricity, gas, steam and air conditionin...",D,"D ELECTRICITY, GAS, STEAM AND AIR CONDITIONING...","""diesel generator operator""","""Operate and manage diesel gensets that will m...","""diesel genset operation"", ""routine inspection...","ANNEX 3: ECONOMIC AND FINANCIAL ANALYSIS: ""The..."


In [61]:
onet_merged_df.columns

Index(['onet_id', 'onet_title', 'onet_description', 'esco_title',
       'esco_description', 'job_zone', 'esco_id'],
      dtype='object')

In [62]:
# Check if esco_id is unique
total_rows = len(onet_merged_df)
unique_esco_ids = onet_merged_df['esco_id'].nunique()
has_duplicates = total_rows != unique_esco_ids

print(f"Total rows: {total_rows}")
print(f"Unique esco_id values: {unique_esco_ids}")
print(f"Has duplicates: {has_duplicates}")

if has_duplicates:
    duplicate_count = total_rows - unique_esco_ids
    print(f"\n⚠️  Found {duplicate_count} duplicate esco_id values")
    
    # Show which esco_ids are duplicated
    duplicated_esco_ids = onet_merged_df[onet_merged_df['esco_id'].duplicated(keep=False)].sort_values('esco_id')
    print(f"\nDuplicated esco_ids ({len(duplicated_esco_ids)} rows):")
    duplicated_esco_ids
else:
    print("\n✓ All esco_id values are unique")

Total rows: 4253
Unique esco_id values: 2652
Has duplicates: True

⚠️  Found 1601 duplicate esco_id values

Duplicated esco_ids (2595 rows):


### 2.02 Merge O*NET Data onto Unique ESCO Occupations

In [63]:
# Prepare onet_merged_df for merge - drop esco_title and esco_description to avoid conflicts
onet_for_merge = onet_merged_df.drop(columns=['esco_title', 'esco_description'])

# Merge O*NET data onto df_occupations by esco_id (left join, allowing duplicates)
df_with_onet = df_occupations.merge(
    onet_for_merge,
    on='esco_id',
    how='left'
)

print(f"✓ Merged O*NET data onto unique ESCO occupations")
print(f"  Original df_occupations rows: {len(df_occupations)}")
print(f"  Merged rows: {len(df_with_onet)} (duplicates expected for multi-mapped ESCO IDs)")
print(f"  Columns: {list(df_with_onet.columns)}")
print(f"\nFirst few rows:")
df_with_onet.head()

✓ Merged O*NET data onto unique ESCO occupations
  Original df_occupations rows: 44
  Merged rows: 79 (duplicates expected for multi-mapped ESCO IDs)
  Columns: ['esco_id', 'esco_label', 'esco_description', 'group_code', 'group_label_en', 'division_code', 'division_label_en', 'section_code', 'section_label_en', 'pad_occupations', 'pad_activities', 'pad_skills', 'pad_quotes', 'onet_id', 'onet_title', 'onet_description', 'job_zone']

First few rows:


,esco_id,esco_label,esco_description,group_code,group_label_en,division_code,division_label_en,section_code,section_label_en,pad_occupations,pad_activities,pad_skills,pad_quotes,onet_id,onet_title,onet_description,job_zone
0,05ebeb56-6ceb-488d-ba91-ed15088efc8e,bridge construction supervisor,Bridge construction supervisors monitor the co...,421,42.1 Construction of roads and railways,42,42 Civil engineering,F,F CONSTRUCTION,"""construction supervisor""","""Supervise construction and installation of fl...","""construction management"", ""site inspection"", ...","ANNEX 4: CLIMATE RISKS 46: ""To enhance the ...",47-1011.00,First-Line Supervisors of Construction Trades ...,Directly supervise and coordinate activities o...,3.0
1,05ebeb56-6ceb-488d-ba91-ed15088efc8e,bridge construction supervisor,Bridge construction supervisors monitor the co...,421,42.1 Construction of roads and railways,42,42 Civil engineering,F,F CONSTRUCTION,"""construction supervisor""","""Supervise construction and installation of fl...","""construction management"", ""site inspection"", ...","ANNEX 4: CLIMATE RISKS 46: ""To enhance the ...",53-6011.00,Bridge and Lock Tenders,"Operate and tend bridges, canal locks, and lig...",2.0
2,0752ed49-03e7-4d75-8314-a051b3771a1d,public administration manager,"Public administration managers direct, monitor...",841,84.1 Administration of the State and the econo...,84,84 Public administration and defence; compulso...,P,P PUBLIC ADMINISTRATION AND DEFENCE; COMPULSOR...,"""head of department""","""Be competitively recruited and take part in o...","""performance management"", ""operational leaders...","V. KEY RISKS: ""Operationalization initiated: ...",11-1021.00,General and Operations Managers,"Plan, direct, or coordinate the operations of ...",4.0
3,0ba06640-e0ac-4911-9e43-289a8e41651e,corporate trainer,"Corporate trainers train, coach, and guide emp...",855,85.5 Other education,85,85 Education,Q,Q EDUCATION,"""capacity building specialist""","""Design and implement capacity-building progra...","""training program design"", ""adult learning tec...","V. KEY RISKS: ""Eligible expenditures include: ...",11-3131.00,Training and Development Managers,"Plan, direct, or coordinate the training and d...",4.0
4,0ba06640-e0ac-4911-9e43-289a8e41651e,corporate trainer,"Corporate trainers train, coach, and guide emp...",855,85.5 Other education,85,85 Education,Q,Q EDUCATION,"""capacity building specialist""","""Design and implement capacity-building progra...","""training program design"", ""adult learning tec...","V. KEY RISKS: ""Eligible expenditures include: ...",13-1151.00,Training and Development Specialists,Design or conduct work-related training and de...,4.0


### 2.03 Take Minimum Job Zone

In [ ]:
# Calculate minimum job zone for each esco_id
df_with_onet['job_zone_min'] = df_with_onet.groupby('esco_id')['job_zone'].transform('min')

# Drop O*NET specific columns and job_zone
columns_to_drop = ['onet_id', 'onet_title', 'onet_description', 'job_zone']
df_final = df_with_onet.drop(columns=columns_to_drop)

# Drop duplicate rows (keep unique esco_id rows)
df_final = df_final.drop_duplicates()

print(f"✓ Calculated minimum job zone and deduplicated")
print(f"  Original df_occupations rows: {len(df_occupations)}")
print(f"  Final rows: {len(df_final)}")
print(f"  Rows match: {len(df_final) == len(df_occupations)}")
print(f"  Columns: {list(df_final.columns)}")
print(f"\nFirst few rows:")
df_final.head()

✓ Calculated minimum job zone and deduplicated
  Original df_occupations rows: 44
  Final rows: 44
  Rows match: True
  Columns: ['esco_id', 'esco_label', 'esco_description', 'group_code', 'group_label_en', 'division_code', 'division_label_en', 'section_code', 'section_label_en', 'pad_occupations', 'pad_activities', 'pad_skills', 'pad_quotes', 'job_zone_min']

First few rows:


,esco_id,esco_label,esco_description,group_code,group_label_en,division_code,division_label_en,section_code,section_label_en,pad_occupations,pad_activities,pad_skills,pad_quotes,job_zone_min
0,05ebeb56-6ceb-488d-ba91-ed15088efc8e,bridge construction supervisor,Bridge construction supervisors monitor the co...,421,42.1 Construction of roads and railways,42,42 Civil engineering,F,F CONSTRUCTION,"""construction supervisor""","""Supervise construction and installation of fl...","""construction management"", ""site inspection"", ...","ANNEX 4: CLIMATE RISKS 46: ""To enhance the ...",2.0
2,0752ed49-03e7-4d75-8314-a051b3771a1d,public administration manager,"Public administration managers direct, monitor...",841,84.1 Administration of the State and the econo...,84,84 Public administration and defence; compulso...,P,P PUBLIC ADMINISTRATION AND DEFENCE; COMPULSOR...,"""head of department""","""Be competitively recruited and take part in o...","""performance management"", ""operational leaders...","V. KEY RISKS: ""Operationalization initiated: ...",4.0
3,0ba06640-e0ac-4911-9e43-289a8e41651e,corporate trainer,"Corporate trainers train, coach, and guide emp...",855,85.5 Other education,85,85 Education,Q,Q EDUCATION,"""capacity building specialist""","""Design and implement capacity-building progra...","""training program design"", ""adult learning tec...","V. KEY RISKS: ""Eligible expenditures include: ...",4.0
5,13d1b2b4-99dd-44da-9734-c9f74bae18f7,customer service representative,Customer service representatives handle compla...,822,82.2 Activities of call centres,82,"82 Office administrative, office support and o...",O,O ADMINISTRATIVE AND SUPPORT SERVICE ACTIVITIES,"""grievance officer""","""Operate the National Grievance Redress Mechan...","""grievance intake"", ""case management"", ""record...","IV. PROJECT APPRAISAL SUMMARY: ""The project wi...",2.0
6,16b974f4-cb8c-436c-8c05-a16957c40131,fossil-fuel power plant operator,Fossil-fuel power plant operators operate and ...,351,"35.1 Electric power generation, transmission a...",35,"35 Electricity, gas, steam and air conditionin...",D,"D ELECTRICITY, GAS, STEAM AND AIR CONDITIONING...","""diesel generator operator""","""Operate and manage diesel gensets that will m...","""diesel genset operation"", ""routine inspection...","ANNEX 3: ECONOMIC AND FINANCIAL ANALYSIS: ""The...",2.0


In [74]:
df_final[['esco_label', 'job_zone_min']].head(50)

,esco_label,job_zone_min
0,bridge construction supervisor,2.0
2,public administration manager,4.0
3,corporate trainer,4.0
5,customer service representative,2.0
6,fossil-fuel power plant operator,2.0
7,consultant social worker,NaN
8,social service consultant,NaN
9,financial auditor,4.0
11,construction engineer,4.0
12,construction quality manager,4.0
